# Classify articles

 # Table of Contents
+ [Import](#Import_0)
+ [File processing](#File_processing_1)
	+ [Load model](#Load_model_2)
	+ [Load raw articles](#Load_raw_articles_3)
		+ [Filter and normalize articles](#Filter_and_normalize_articles_4)
+ [Classify articles](#Classify_articles_5)
+ [Results](#Results_6)


<a class="anchor" id="Import_0"></a>
# <span style="color: #ff2a00">Import</span>

In [1]:
import sys
sys.path.append("..")
import pickle
import pandas as pd

from text_analysis import *

from file_management import check_save_file, get_files_dir
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()


In [2]:
import warnings
warnings.filterwarnings('ignore', message='Liblinear failed to converge, increase the number of iterations.')

import sys
sys.path.append("../")
sys.path.append("../Classify_articles/")
sys.path.append("../Extract_information/")


import pandas as pd
import numpy as np
from random import sample

from file_management import check_save_file, get_files_dir


from text_analysis import normalizar, normalize_org, remove_stopwords, normalize_chemical
from plot_models import *

# Output_files

In [ ]:
general_name = 'full_text_classified_w_prod'

today = date.today()
today = today.strftime("%Y_%m_%d")

classified_articles_file = f'{general_name}_V_{today}.json'
classified_articles_file

classified_articles_file = 'classified_articles_v_2025_06_30.csv'

<a class="anchor" id="File_processing_1"></a>
# <span style="color: #ff2a00">File processing</span>

<a class="anchor" id="Load_model_2"></a>
## <span style="color: #ff5500">Load model</span>

The model we will use to classify the data 

In [24]:
models = pickle.load(open('./model_3.pickle', 'rb'))
meta_model = pickle.load(open('./meta_model_3.pickle', 'rb'))


In [25]:
models

,Model,Model_file,True Positive (%),True Negative (%),AUC,Vectorizer,Transformer,Pipeline,Trained_on,TP,TN,FP,FN,Sensitivity,Specificity,Model_pipeline
3,SVC,"SVC(class_weight='balanced', kernel='linear', ...",83.095238,92.255125,0.876752,TfidfVectorizer(),TfidfTransformer(),vec2 + transformer,Both,83.095238,92.255125,16.904762,7.744875,0.914742,0.845138,SVC vec2 + transformer Both
0,MultinomialNB,"MultinomialNB(alpha=21.0, class_prior=[0.2, 0.8])",82.619048,92.482916,0.875510,CountVectorizer(max_df=0.85),None,Vectorizer Only,Both,82.619048,92.482916,17.380952,7.517084,0.916603,0.841796,MultinomialNB Vectorizer Only Both


In [17]:
modelito = meta_model.iloc[0][0]

In [19]:
modelito.classes_

array([0, 1])

In [23]:
modelito.__dict__

{'penalty': 'l2',
 'dual': False,
 'tol': 0.0001,
 'C': 1.0,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'class_weight': None,
 'random_state': None,
 'solver': 'lbfgs',
 'max_iter': 100,
 'multi_class': 'deprecated',
 'verbose': 0,
 'warm_start': False,
 'n_jobs': None,
 'l1_ratio': None,
 'n_features_in_': 12,
 'classes_': array([0, 1]),
 'n_iter_': array([15], dtype=int32),
 'coef_': array([[0.        , 0.        , 0.        , 0.        , 3.76781145,
         4.06064532, 0.90033829, 0.        , 0.        , 0.        ,
         0.        , 0.        ]]),
 'intercept_': array([-4.8237849])}

<a class="anchor" id="Load_raw_articles_3"></a>
## <span style="color: #ff5500">Load raw articles</span>

In [12]:
file = OUTPUT_DIR+'/Articles/raw_articles.csv'
first_filtered_data = pd.read_csv(file, sep=",")


/var/folders/v6/y2v6wwn93yj5vvhl558fx2zc0000gn/T/ipykernel_5144/2364390065.py:2: DtypeWarning: Columns (1,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  first_filtered_data = pd.read_csv(file, sep=",")


<a class="anchor" id="Filter_and_normalize_articles_4"></a>
### <span style="color: #ff8000">Filter and normalize articles</span>

Remove the articles that did not have a title nor abstract

In [17]:
first_filtered_data = first_filtered_data.loc[~first_filtered_data.Title.isna()]
first_filtered_data = first_filtered_data.loc[~first_filtered_data.Abstract.isna()]

In [18]:
len(first_filtered_data)

152921

Normalize data (remove organism names or other common words)

In [19]:
# Remove stopwords
first_filtered_data.loc[:,'Abstract_norm'] = first_filtered_data.loc[:,'Abstract'].map(remove_stopwords)
first_filtered_data.loc[:,'Title_norm'] = first_filtered_data.loc[:,'Title'].map(remove_stopwords)

# Normalize overrepresentated organism
first_filtered_data.loc[:,'Abstract_norm'] = first_filtered_data.loc[:,'Abstract'].map(normalize_org)
first_filtered_data.loc[:,'Title_norm'] = first_filtered_data.loc[:,'Title'].map(normalize_org)

# Normalize text 
first_filtered_data.loc[:,'Abstract_norm'] = first_filtered_data.loc[:,'Abstract'].map(normalizar)
first_filtered_data.loc[:,'Title_norm'] = first_filtered_data.loc[:,'Title'].map(normalizar)

In [20]:
print('miau')

miau


In [21]:
data_to_classify = first_filtered_data.loc[:,'Title_norm'] +' '+ first_filtered_data.loc[:,'Abstract_norm']

<a class="anchor" id="Classify_articles_5"></a>
# <span style="color: #ff2a00">Classify articles</span>

In [22]:
def predict_with_meta_model(models, meta_model, raw_title, raw_abstract, raw_both):
    meta_features_test = []

    # Iterate through each model and make predictions
    for _,model in models.iterrows():
        vectorizer = model['Vectorizer']
        transformer = model['Transformer']
        clf = model['Model_file']

        # Transform the new data
        X_test_title, X_test_abstract, X_test_both, pipeline = transform_LASER_data(
            vectorizer, transformer, raw_title, raw_abstract, raw_both)

        # Generate predictions (or probabilities) from the base models
        preds_test_title = clf.predict_proba(X_test_title)[:, 1]
        preds_test_abstract = clf.predict_proba(X_test_abstract)[:, 1]
        preds_test_both = clf.predict_proba(X_test_both)[:, 1]

        # Combine predictions into meta-features
        meta_features_test.append(np.column_stack((preds_test_title, preds_test_abstract, preds_test_both)))

    # Stack all model features horizontally to create the final meta-feature set
    X_meta_test = np.hstack(meta_features_test)

    # Use the meta-model to make final predictions
    final_predictions = meta_model.predict(X_meta_test)

    return final_predictions

In [23]:
meta_model=meta_model.iloc[0]

In [24]:
meta_model = meta_model.iloc[0]

In [25]:
meta_model

LogisticRegression()

In [26]:


# New data to predict
raw_title_test = first_filtered_data.loc[:,'Title_norm']
raw_abstract_test = first_filtered_data.loc[:,'Abstract_norm']
raw_both_test = data_to_classify

# Predict using the meta-model
final_predictions = predict_with_meta_model(models, meta_model, raw_title_test, raw_abstract_test, raw_both_test)

# Now, `final_predictions` contains the final predicted classes
print(final_predictions)

[0 0 0 ... 0 0 1]


In [27]:
first_filtered_data.loc[:,'Classification'] = final_predictions

In [29]:
classified_data = first_filtered_data.loc[first_filtered_data.Classification==1,:]
classified_data.head()

,Unnamed: 0,PM_ID,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Abstract_norm,Title_norm,Classification
13,13,10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",the psbaii locu wa use as an integr platform t...,increas product of zeaxanthin and other pigmen...,1
14,14,10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",express of the vhb gene encod hemoglobin from ...,express of alcaligen eutrophu flavohemoprotein...,1
31,31,10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,NaN,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",there is an increas interest in environment bi...,environment biotechnolog .,1
50,50,10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",glycolyt flux in rest escherichia coli were en...,alter regul of pyruv kinas or co-overexpress o...,1
51,51,10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",the squalen synthas ( sq ) gene encod a key re...,clone and character of the yarrowia lipolytica...,1


In [30]:
classified_data = classified_data.drop(columns=['Title_norm', 'Abstract_norm'])

In [31]:
len(classified_data)

20108

<a class="anchor" id="Results_6"></a>
# <span style="color: #ff2a00">Results</span>

In [32]:
classified_data.Journal.value_counts().head(10)

Journal
Metabolic engineering                         1375
Microbial cell factories                      1149
Applied microbiology and biotechnology        1107
Bioresource technology                         805
ACS synthetic biology                          746
Applied and environmental microbiology         645
Journal of agricultural and food chemistry     626
Biotechnology and bioengineering               613
Biotechnology for biofuels                     603
Journal of biotechnology                       427
Name: count, dtype: int64

In [54]:
check_save_file(classified_data, classified_articles_file, 'Articles')

Saved file in: /Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Output/Articles/classified_articles_v_2025_06_30.csv
